# Optional: BB84 quantum key distribution

**Goal:** Alice and Bob build a shared secret key, and an eavesdropper
reveals herself through the errors she causes.

1. Alice picks a random bit and a random basis (Z or X) and sends a qubit.
2. Bob measures in his own random basis.
3. They publicly compare *bases* (not bits) and keep only the positions
   where the bases matched.
4. They compare a sample of the kept bits. Errors mean someone measured
   the qubits in transit.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ms-cc-org/qc-workshop/blob/main/solutions/06_bb84_optional.ipynb)

**Solution notebook.**

In [ ]:
# Install pinned versions (takes about a minute in Colab; already done locally)
%pip install -q qiskit==2.6.1 qiskit-aer==0.18.2 qiskit-ibm-runtime==0.51.0 matplotlib pylatexenc

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

sim = AerSimulator()

def run(qc, shots=1000, backend=sim):
    """Run a circuit and return the measurement counts."""
    return backend.run(transpile(qc, backend), shots=shots).result().get_counts()

import random

N = 200  # qubits Alice sends

In [ ]:
def bb84_round(alice_bit, alice_basis, bob_basis, eve_basis=None):
    """Send one qubit. Bases: 0 = Z (standard), 1 = X (Hadamard)."""
    qc = QuantumCircuit(1, 1)
    if alice_bit:
        qc.x(0)
    if alice_basis:
        qc.h(0)
    if eve_basis is not None:
        eavesdrop(qc, eve_basis)
    if bob_basis:
        qc.h(0)
    qc.measure(0, 0)
    return qc

def eavesdrop(qc, eve_basis):
    pass  # replaced in Exercise 6.1

def run_protocol(with_eve=False):
    alice_bits = [random.randint(0, 1) for _ in range(N)]
    alice_bases = [random.randint(0, 1) for _ in range(N)]
    bob_bases = [random.randint(0, 1) for _ in range(N)]
    eve_bases = [random.randint(0, 1) if with_eve else None for _ in range(N)]

    circuits = [bb84_round(*args) for args in zip(alice_bits, alice_bases, bob_bases, eve_bases)]
    results = sim.run(transpile(circuits, sim), shots=1, memory=True).result()
    bob_bits = [int(results.get_memory(i)[0]) for i in range(N)]

    kept = [i for i in range(N) if alice_bases[i] == bob_bases[i]]
    errors = sum(alice_bits[i] != bob_bits[i] for i in kept)
    return len(kept), errors

kept, errors = run_protocol()
print(f"Kept {kept} of {N} bits, errors: {errors}")

With no eavesdropper there are no errors, so the kept bits form a shared
secret key. About half of the bits survive, because the bases match
half the time.

## Exercise 6.1: Add Eve

Eve intercepts each qubit, measures it in her own random basis, and sends
on what she saw. Fill in `eavesdrop`: rotate into her basis, measure,
then rotate back so the qubit continues in her basis.

In [ ]:
def eavesdrop(qc, eve_basis):
    if eve_basis:
        qc.h(0)
    qc.measure(0, 0)
    if eve_basis:
        qc.h(0)

kept, errors = run_protocol(with_eve=True)
print(f"Kept {kept} bits, errors: {errors} ({errors / kept:.0%})")
assert 0.1 < errors / kept < 0.4

Eve guesses the wrong basis half the time, and each wrong guess gives
Bob a random bit, so about 25% of the kept bits disagree. Alice and Bob
compare a sample, see the errors and throw the key away. Eve cannot
measure without leaving a trace: that is the security guarantee.